# Time Travel Demo

Time travel lets you query, clone, or restore data as it existed at any point within a retention period (default 1 day, up to 90 days). Useful for recovering from accidental changes or auditing data history.

In [ ]:
%%sql -r ctx
use role sysadmin;
use warehouse compute_wh;

In [ ]:
%%sql -r schema_result
create database if not exists demo;
create schema if not exists demo.time_travel;

In [ ]:
%%sql -r create_table
create or replace table demo.time_travel.products (
    product_id int primary key,
    product_name varchar(100),
    price number(10,2),
    category varchar(50)
);
insert into demo.time_travel.products (product_id, product_name, price, category) values
    (1, 'Laptop', 999.99, 'Electronics'),
    (2, 'Headphones', 79.99, 'Electronics'),
    (3, 'Notebook', 4.99, 'Office'),
    (4, 'Desk Lamp', 34.99, 'Office'),
    (5, 'Backpack', 49.99, 'Accessories');
select current_timestamp();

In [ ]:
%%sql -r current_state
select * from demo.time_travel.products;

In [ ]:
%%sql -r save_qid
set query_id = last_query_id();

In [ ]:
%%sql -r bad_update
update demo.time_travel.products set price = 0 where category = 'Electronics';
select current_timestamp();

In [ ]:
%%sql -r verify_damage
select * from demo.time_travel.products;

In [ ]:
%%sql -r offset_query
select * from demo.time_travel.products at(offset => -60);

In [ ]:
%%sql -r before_query
select * from demo.time_travel.products before(statement => $query_id);

In [ ]:
%%sql -r clone_result
create or replace table demo.time_travel.products_restored
    clone demo.time_travel.products before(statement => $query_id);

In [ ]:
%%sql -r restored_data
select * from demo.time_travel.products_restored;

In [ ]:
%%sql -r fix_original
create or replace table demo.time_travel.products
    clone demo.time_travel.products_restored;

In [ ]:
%%sql -r fixed_data
select * from demo.time_travel.products;

# UNDROP

What happens if we mistakenly drop a table? Can we recover it?

In [ ]:
%%sql -r drop_result
drop table demo.time_travel.products_restored;

In [ ]:
%%sql -r undrop_result
undrop table demo.time_travel.products_restored;

In [ ]:
%%sql -r undrop_data
select * from demo.time_travel.products_restored;

## Retention Period

In [ ]:
%%sql -r retention_result
alter table demo.time_travel.products set data_retention_time_in_days = 7;

In [ ]:
%%sql -r show_tables
show tables like 'PRODUCTS' in schema demo.time_travel;

## Cleanup

In [ ]:
%%sql -r cleanup
drop table if exists demo.time_travel.products_restored;
drop table if exists demo.time_travel.products;
drop schema if exists demo.time_travel;